# Marathos EDA from bronze layer to silver layer cleaning (ATHLETE)

```
athlete_performance
athlete_club
athlete_country
athlete_year_of_birth
athlete_age_category
athlete_gender
athlete_average_speed
athlete_id
```


## Load bronze and rename columns

In [0]:
# imports
import re
from pyspark.sql.functions import col, when, count, upper, to_date, try_to_date, expr, regexp_extract, concat, lit, trim, length, lower, regexp_replace, coalesce, countDistinct



In [0]:
# Load Bronze marathon table
marathon_bronze_df = spark.sql("""
    SELECT *
    FROM marathos_catalog.bronze.raw_marathon_results
""")

# Convert raw column names to snake_case
def to_snake_case(column_name):
    clean_name = column_name.strip().lower()
    clean_name = re.sub(r"[^a-z0-9]+", "_", clean_name)
    return clean_name.strip("_")

def rename_columns_to_snake_case(df):
    new_columns = [to_snake_case(column) for column in df.columns]
    return df.toDF(*new_columns)

marathon_silver_df = rename_columns_to_snake_case(marathon_bronze_df)

print(f"Starting rows: {marathon_silver_df.count()}")

In [0]:
# Check schema
marathon_silver_df.columns

## 1. athlete_performance
**athlete_performance has mixed formats based on raw level eda**
- time values, like 8:52:00 h
- distance values, like 160.944 km
- 2 null rows

**and it needs to match the `event_distance_length` column's results:**
- if event unit is km or mi → athlete_performance should be time
- if event unit is h → athlete_performance should be distance

#### recreate `event-cleaned` marathon_silver_df
- I need to reapply the `event_distance_length` cleaning I did on the other ntbk to be able to clean and match `athlete_performance`.

In [0]:
# This is the cleaning already tested in the event notebook.
# Keep original event date and classify date format
marathon_silver_df = (
    marathon_silver_df
    .withColumn("event_date_raw", col("event_dates"))
    .withColumn(
        "event_date_format_type",
        when(col("event_dates").rlike(r"^\d{2}\.\d{2}\.\d{4}$"), "single_date")
        .when(col("event_dates").rlike(r"^\d{2}\.-\d{2}\.\d{2}\.\d{4}$"), "date_range_same_month")
        .when(col("event_dates").rlike(r"^\d{2}\.\d{2}\.-\d{2}\.\d{2}\.\d{4}$"), "date_range_different_month")
        .when(col("event_dates").rlike(r"^\d{2}\.\d{2}\.\d{4}-\d{2}\.\d{2}\.\d{4}$"), "date_range_different_year")
        .otherwise("unknown_format")
    )
)

# Parse single-date events
marathon_silver_df = marathon_silver_df.withColumn(
    "event_start_date",
    when(
        col("event_date_format_type") == "single_date",
        expr("try_to_date(event_dates, 'dd.MM.yyyy')")
    )
)

# Same-month ranges, example: 23.-24.11.1991
same_month_start_day = regexp_extract(col("event_dates"), r"^(\d{2})\.-(\d{2})\.(\d{2})\.(\d{4})$", 1)
same_month_end_day = regexp_extract(col("event_dates"), r"^(\d{2})\.-(\d{2})\.(\d{2})\.(\d{4})$", 2)
same_month_month = regexp_extract(col("event_dates"), r"^(\d{2})\.-(\d{2})\.(\d{2})\.(\d{4})$", 3)
same_month_year = regexp_extract(col("event_dates"), r"^(\d{2})\.-(\d{2})\.(\d{2})\.(\d{4})$", 4)

marathon_silver_df = (
    marathon_silver_df
    .withColumn("same_month_start_raw", concat(same_month_start_day, lit("."), same_month_month, lit("."), same_month_year))
    .withColumn("same_month_end_raw", concat(same_month_end_day, lit("."), same_month_month, lit("."), same_month_year))
)

marathon_silver_df = (
    marathon_silver_df
    .withColumn(
        "event_start_date",
        when(
            col("event_date_format_type") == "date_range_same_month",
            expr("try_to_date(same_month_start_raw, 'dd.MM.yyyy')")
        ).otherwise(col("event_start_date"))
    )
    .withColumn(
        "event_end_date",
        when(col("event_date_format_type") == "single_date", col("event_start_date"))
        .when(
            col("event_date_format_type") == "date_range_same_month",
            expr("try_to_date(same_month_end_raw, 'dd.MM.yyyy')")
        )
    )
)

# Different-month ranges, example: 30.10.-03.11.1991
diff_month_start_day = regexp_extract(col("event_dates"), r"^(\d{2})\.(\d{2})\.-(\d{2})\.(\d{2})\.(\d{4})$", 1)
diff_month_start_month = regexp_extract(col("event_dates"), r"^(\d{2})\.(\d{2})\.-(\d{2})\.(\d{2})\.(\d{4})$", 2)
diff_month_end_day = regexp_extract(col("event_dates"), r"^(\d{2})\.(\d{2})\.-(\d{2})\.(\d{2})\.(\d{4})$", 3)
diff_month_end_month = regexp_extract(col("event_dates"), r"^(\d{2})\.(\d{2})\.-(\d{2})\.(\d{2})\.(\d{4})$", 4)
diff_month_end_year = regexp_extract(col("event_dates"), r"^(\d{2})\.(\d{2})\.-(\d{2})\.(\d{2})\.(\d{4})$", 5)

marathon_silver_df = (
    marathon_silver_df
    .withColumn(
        "diff_month_start_raw",
        concat(
            diff_month_start_day,
            lit("."),
            diff_month_start_month,
            lit("."),
            when(
                diff_month_start_month.cast("int") > diff_month_end_month.cast("int"),
                diff_month_end_year.cast("int") - 1
            ).otherwise(diff_month_end_year)
        )
    )
    .withColumn("diff_month_end_raw", concat(diff_month_end_day, lit("."), diff_month_end_month, lit("."), diff_month_end_year))
)

marathon_silver_df = (
    marathon_silver_df
    .withColumn(
        "event_start_date",
        when(
            col("event_date_format_type") == "date_range_different_month",
            expr("try_to_date(diff_month_start_raw, 'dd.MM.yyyy')")
        ).otherwise(col("event_start_date"))
    )
    .withColumn(
        "event_end_date",
        when(
            col("event_date_format_type") == "date_range_different_month",
            expr("try_to_date(diff_month_end_raw, 'dd.MM.yyyy')")
        ).otherwise(col("event_end_date"))
    )
)

# Different-year ranges, example: 31.12.1992-01.01.1993
diff_year_start_day = regexp_extract(col("event_dates"), r"^(\d{2})\.(\d{2})\.(\d{4})-(\d{2})\.(\d{2})\.(\d{4})$", 1)
diff_year_start_month = regexp_extract(col("event_dates"), r"^(\d{2})\.(\d{2})\.(\d{4})-(\d{2})\.(\d{2})\.(\d{4})$", 2)
diff_year_start_year = regexp_extract(col("event_dates"), r"^(\d{2})\.(\d{2})\.(\d{4})-(\d{2})\.(\d{2})\.(\d{4})$", 3)
diff_year_end_day = regexp_extract(col("event_dates"), r"^(\d{2})\.(\d{2})\.(\d{4})-(\d{2})\.(\d{2})\.(\d{4})$", 4)
diff_year_end_month = regexp_extract(col("event_dates"), r"^(\d{2})\.(\d{2})\.(\d{4})-(\d{2})\.(\d{2})\.(\d{4})$", 5)
diff_year_end_year = regexp_extract(col("event_dates"), r"^(\d{2})\.(\d{2})\.(\d{4})-(\d{2})\.(\d{2})\.(\d{4})$", 6)

marathon_silver_df = (
    marathon_silver_df
    .withColumn("diff_year_start_raw", concat(diff_year_start_day, lit("."), diff_year_start_month, lit("."), diff_year_start_year))
    .withColumn("diff_year_end_raw", concat(diff_year_end_day, lit("."), diff_year_end_month, lit("."), diff_year_end_year))
)

marathon_silver_df = (
    marathon_silver_df
    .withColumn(
        "event_start_date",
        when(
            col("event_date_format_type") == "date_range_different_year",
            expr("try_to_date(diff_year_start_raw, 'dd.MM.yyyy')")
        ).otherwise(col("event_start_date"))
    )
    .withColumn(
        "event_end_date",
        when(
            col("event_date_format_type") == "date_range_different_year",
            expr("try_to_date(diff_year_end_raw, 'dd.MM.yyyy')")
        ).otherwise(col("event_end_date"))
    )
)

# Remove invalid parsed dates
marathon_silver_df = marathon_silver_df.filter(
    col("event_start_date").isNotNull() &
    col("event_end_date").isNotNull()
)

#### Reapply event distance and finisher cleaning

In [0]:
# Classify event distance type
marathon_silver_df = marathon_silver_df.withColumn(
    "event_distance_type",
    when(lower(trim(col("event_distance_length"))).rlike(r"^[0-9]+(\.[0-9]+)?km$"), "distance_km")
    .when(lower(trim(col("event_distance_length"))).rlike(r"^[0-9]+(\.[0-9]+)?mi$"), "distance_mi")
    .when(lower(trim(col("event_distance_length"))).rlike(r"^[0-9]+h$"), "time_hours")
    .when(lower(trim(col("event_distance_length"))).rlike(r"^[0-9]+(\.[0-9]+)?k$"), "distance_k_needs_standardization")
    .when(lower(trim(col("event_distance_length"))).rlike(r"^[0-9]+d$"), "days_invalid")
    .when(lower(trim(col("event_distance_length"))).contains("etappen"), "multi_stage_invalid")
    .otherwise("invalid_or_unknown")
)

# Keep only valid event distance types
marathon_silver_df = marathon_silver_df.filter(
    col("event_distance_type").isin(
        "distance_km",
        "distance_mi",
        "time_hours",
        "distance_k_needs_standardization"
    )
)

# Standardize k/K to km and lowercase all event_distance_length values
marathon_silver_df = marathon_silver_df.withColumn(
    "event_distance_length",
    when(
        col("event_distance_type") == "distance_k_needs_standardization",
        regexp_replace(lower(trim(col("event_distance_length"))), "k$", "km")
    ).otherwise(lower(trim(col("event_distance_length"))))
)

# Reclassify standardized k/K values as distance_km
marathon_silver_df = marathon_silver_df.withColumn(
    "event_distance_type",
    when(col("event_distance_type") == "distance_k_needs_standardization", "distance_km")
    .otherwise(col("event_distance_type"))
)

# Extract numeric event value and unit
marathon_silver_df = (
    marathon_silver_df
    .withColumn(
        "event_distance_value",
        regexp_extract(col("event_distance_length"), r"^([0-9]+(\.[0-9]+)?)", 1).cast("double")
    )
    .withColumn(
        "event_distance_unit",
        regexp_extract(col("event_distance_length"), r"(km|mi|h)$", 1)
    )
)

# Remove rows with 0 or missing finishers
marathon_silver_df = marathon_silver_df.filter(
    col("event_number_of_finishers").isNotNull() &
    (col("event_number_of_finishers") > 0)
)

# Drop temporary date helper columns
marathon_silver_df = marathon_silver_df.drop(
    "same_month_start_raw",
    "same_month_end_raw",
    "diff_month_start_raw",
    "diff_month_end_raw",
    "diff_year_start_raw",
    "diff_year_end_raw"
)

print(f"Rows after event baseline cleaning: {marathon_silver_df.count()}")

#### classify athlete_performance

In [0]:

# Classify athlete_performance values by format
athlete_performance_type_df = (
    marathon_silver_df
    .withColumn(
        "athlete_performance_type",
        when(col("athlete_performance").isNull(), "null")
        .when(trim(col("athlete_performance")).rlike(r"^[0-9]+:[0-9]{2}:[0-9]{2} h$"), "time_hours")
        .when(trim(col("athlete_performance")).rlike(r"^[0-9]+(\.[0-9]+)? km$"), "distance_km")
        .when(trim(col("athlete_performance")).rlike(r"^[0-9]+(\.[0-9]+)? mi$"), "distance_mi")
        .otherwise("unknown")
    )
    .groupBy("athlete_performance_type")
    .agg(count("*").alias("row_count"))
    .orderBy(col("row_count").desc())
)

display(athlete_performance_type_df)

#### inspect unknown/null performance rows

- The unknown group is not automatically invalid. I also have results in these formats:
```
13d 14:02:52 h
14d 02:07:33 h
17d 06:38:56 h
```

- These are still time-based performances, just longer than normal because they include days. My first regex only accepted this pattern:
```
8:52:00 h
```
But not this: 
```
13d 14:02:52 h
```

**Next Step is to update the classification before removing anything.**
-If the event unit is km or mi, athlete performance should be in time.
- if the event unit is h, performance should be in distance.
- So these 13d ... h values are valid for long mi/km events because they are still time durations.

In [0]:
# Inspect rows where athlete_performance is null or unknown
invalid_athlete_performance_df = (
    marathon_silver_df
    .withColumn(
        "athlete_performance_type",
        when(col("athlete_performance").isNull(), "null")
        .when(trim(col("athlete_performance")).rlike(r"^[0-9]+:[0-9]{2}:[0-9]{2} h$"), "time_hours")
        .when(trim(col("athlete_performance")).rlike(r"^[0-9]+(\.[0-9]+)? km$"), "distance_km")
        .when(trim(col("athlete_performance")).rlike(r"^[0-9]+(\.[0-9]+)? mi$"), "distance_mi")
        .otherwise("unknown")
    )
    .filter(col("athlete_performance_type").isin("null", "unknown"))
    .select(
        "year_of_event",
        "event_dates",
        "event_name",
        "event_distance_length",
        "event_distance_unit",
        "athlete_performance",
        "athlete_country",
        "athlete_id"
    )
)

display(invalid_athlete_performance_df)

#### classify day-based time performance properly
- the unknown count should drop a lot, probably close to 0.

In [0]:
# Classify athlete_performance values by format
athlete_performance_type_df = (
    marathon_silver_df
    .withColumn(
        "athlete_performance_type",
        when(col("athlete_performance").isNull(), "null")
        .when(trim(col("athlete_performance")).rlike(r"^[0-9]+:[0-9]{2}:[0-9]{2} h$"), "time_hours")
        .when(trim(col("athlete_performance")).rlike(r"^[0-9]+d [0-9]{2}:[0-9]{2}:[0-9]{2} h$"), "time_days_hours")
        .when(trim(col("athlete_performance")).rlike(r"^[0-9]+(\.[0-9]+)? km$"), "distance_km")
        .when(trim(col("athlete_performance")).rlike(r"^[0-9]+(\.[0-9]+)? mi$"), "distance_mi")
        .otherwise("unknown")
    )
    .groupBy("athlete_performance_type")
    .agg(count("*").alias("row_count"))
    .orderBy(col("row_count").desc())
)

display(athlete_performance_type_df)

#### inspect anything still unknown

In [0]:
# Inspect remaining unknown or null athlete_performance values
invalid_athlete_performance_df = (
    marathon_silver_df
    .withColumn(
        "athlete_performance_type",
        when(col("athlete_performance").isNull(), "null")
        .when(trim(col("athlete_performance")).rlike(r"^[0-9]+:[0-9]{2}:[0-9]{2} h$"), "time_hours")
        .when(trim(col("athlete_performance")).rlike(r"^[0-9]+d [0-9]{2}:[0-9]{2}:[0-9]{2} h$"), "time_days_hours")
        .when(trim(col("athlete_performance")).rlike(r"^[0-9]+(\.[0-9]+)? km$"), "distance_km")
        .when(trim(col("athlete_performance")).rlike(r"^[0-9]+(\.[0-9]+)? mi$"), "distance_mi")
        .otherwise("unknown")
    )
    .filter(col("athlete_performance_type").isin("null", "unknown"))
    .select(
        "year_of_event",
        "event_dates",
        "event_name",
        "event_distance_length",
        "event_distance_unit",
        "athlete_performance",
        "athlete_country",
        "athlete_id"
    )
)

display(invalid_athlete_performance_df)

- The remaining invalid group is small: `unknown + null = 26 rows`
- **Remove athlete_performance rows where type is null or unknown.**

#### add athlete_performance_type to the main dataframe

In [0]:
# Add athlete_performance_type to the main dataframe
marathon_silver_df = marathon_silver_df.withColumn(
    "athlete_performance_type",
    when(col("athlete_performance").isNull(), "null")
    .when(trim(col("athlete_performance")).rlike(r"^[0-9]+:[0-9]{2}:[0-9]{2} h$"), "time_hours")
    .when(trim(col("athlete_performance")).rlike(r"^[0-9]+d [0-9]{2}:[0-9]{2}:[0-9]{2} h$"), "time_days_hours")
    .when(trim(col("athlete_performance")).rlike(r"^[0-9]+(\.[0-9]+)? km$"), "distance_km")
    .when(trim(col("athlete_performance")).rlike(r"^[0-9]+(\.[0-9]+)? mi$"), "distance_mi")
    .otherwise("unknown")
)

#### remove null and unknown performance rows

In [0]:
# Remove rows with null or unknown athlete_performance
rows_before_performance_cleaning = marathon_silver_df.count()

marathon_silver_df = marathon_silver_df.filter(
    ~col("athlete_performance_type").isin("null", "unknown")
)

rows_after_performance_cleaning = marathon_silver_df.count()

print(f"Rows before athlete_performance cleaning: {rows_before_performance_cleaning}")
print(f"Rows after athlete_performance cleaning: {rows_after_performance_cleaning}")
print(f"Rows removed: {rows_before_performance_cleaning - rows_after_performance_cleaning}")

#### validate event unit vs performance type

In [0]:
# Validate event unit against athlete performance type
invalid_event_performance_match_df = (
    marathon_silver_df
    .filter(
        (
            col("event_distance_unit").isin("km", "mi") &
            ~col("athlete_performance_type").isin("time_hours", "time_days_hours")
        )
        |
        (
            (col("event_distance_unit") == "h") &
            ~col("athlete_performance_type").isin("distance_km", "distance_mi")
        )
    )
)

print(f"Invalid event/performance match rows: {invalid_event_performance_match_df.count()}")

#### Results:
The athlete_performance column contains both time-based and distance-based values.

After classification, I removed null and unknown performance rows. The remaining values are valid performance formats:
- time_hours
- time_days_hours
- distance_km

The performance type also matches the event type rule:
- km/mi events have time-based performance
- h events have distance-based performance

In [0]:
# Display athlete_performance with its classification
display(
    marathon_silver_df
    .select(
        "event_distance_length",
        "event_distance_unit",
        "athlete_performance",
        "athlete_performance_type"
    )
    .limit(100)
)

In [0]:
# Count cleaned athlete_performance types
display(
    marathon_silver_df
    .groupBy("athlete_performance_type")
    .agg(count("*").alias("row_count"))
    .orderBy("athlete_performance_type")
)

#### parse athlete_performance into numeric analysis columns

In [0]:
# Extract time parts from normal time format, example: 7:49:37 h
performance_hours = regexp_extract(col("athlete_performance"), r"^([0-9]+):([0-9]{2}):([0-9]{2}) h$", 1)
performance_minutes = regexp_extract(col("athlete_performance"), r"^([0-9]+):([0-9]{2}):([0-9]{2}) h$", 2)
performance_seconds = regexp_extract(col("athlete_performance"), r"^([0-9]+):([0-9]{2}):([0-9]{2}) h$", 3)

# Extract time parts from day + time format, example: 13d 14:02:52 h
performance_days = regexp_extract(col("athlete_performance"), r"^([0-9]+)d ([0-9]{2}):([0-9]{2}):([0-9]{2}) h$", 1)
performance_day_hours = regexp_extract(col("athlete_performance"), r"^([0-9]+)d ([0-9]{2}):([0-9]{2}):([0-9]{2}) h$", 2)
performance_day_minutes = regexp_extract(col("athlete_performance"), r"^([0-9]+)d ([0-9]{2}):([0-9]{2}):([0-9]{2}) h$", 3)
performance_day_seconds = regexp_extract(col("athlete_performance"), r"^([0-9]+)d ([0-9]{2}):([0-9]{2}):([0-9]{2}) h$", 4)

# Convert time-based performance to total seconds
marathon_silver_df = marathon_silver_df.withColumn(
    "athlete_performance_seconds",
    when(
        col("athlete_performance_type") == "time_hours",
        performance_hours.cast("int") * 3600
        + performance_minutes.cast("int") * 60
        + performance_seconds.cast("int")
    ).when(
        col("athlete_performance_type") == "time_days_hours",
        performance_days.cast("int") * 86400
        + performance_day_hours.cast("int") * 3600
        + performance_day_minutes.cast("int") * 60
        + performance_day_seconds.cast("int")
    )
)

# Extract distance-based performance value for fixed-hour events
marathon_silver_df = marathon_silver_df.withColumn(
    "athlete_performance_distance",
    when(
        col("athlete_performance_type") == "distance_km",
        regexp_extract(col("athlete_performance"), r"^([0-9]+(\.[0-9]+)?) km$", 1).cast("double")
    )
)

# Store the unit of the parsed performance value
marathon_silver_df = marathon_silver_df.withColumn(
    "athlete_performance_unit",
    when(col("athlete_performance_type").isin("time_hours", "time_days_hours"), "seconds")
    .when(col("athlete_performance_type") == "distance_km", "km")
)

#### display the parsed result

In [0]:
# Display parsed athlete_performance result
display(
    marathon_silver_df
    .select(
        "event_distance_length",
        "event_distance_unit",
        "athlete_performance",
        "athlete_performance_type",
        "athlete_performance_seconds",
        "athlete_performance_distance",
        "athlete_performance_unit"
    )
)

In [0]:
# Check if any valid performance rows failed to parse
invalid_performance_parse_count = (
    marathon_silver_df
    .filter(
        (
            col("athlete_performance_type").isin("time_hours", "time_days_hours") &
            col("athlete_performance_seconds").isNull()
        )
        |
        (
            (col("athlete_performance_type") == "distance_km") &
            col("athlete_performance_distance").isNull()
        )
    )
    .count()
)

print(f"Invalid athlete_performance parse rows: {invalid_performance_parse_count}")

#### Results:
Created new columns:
```
athlete_performance_type
athlete_performance_seconds
athlete_performance_distance
athlete_performance_unit
```

**Distance-based events: 60km, 100km, 28mi^**
- athlete_performance is time
- athlete_performance_seconds has value
- athlete_performance_distance is NULL

**Time-based events: 6h, 12h, 24h**
- athlete_performance is distance completed
- athlete_performance_distance has value
- athlete_performance_seconds is NULL

----

- The athlete_performance column can represent either time or distance depending on the event type.

- For distance-based events such as km or mi races, athlete_performance is converted into athlete_performance_seconds.

- For time-based events such as 6h, 12h, or 24h races, athlete_performance is converted into athlete_performance_distance.

- Because these are two different measurement types, one of the parsed columns will be NULL depending on the event type. These NULL values are expected and should not be replaced with "unknown" because the columns are numeric and need to support calculations later.

## 2. `athlete_club`
- the Silver rule for athlete_club should be light text standardization, not row removal.
- **Findings from raw layer.**
    - Null athlete_club rows: 2,826,373 in raw data
    - Null percentage: 37.88%
    - Distinct club values: 552,190
    - Many values start with *
    - Some values contain non-ASCII characters

- **So in silver layer:**
1. Do not remove rows where athlete_club is missing.
2. Replace null/empty athlete_club with "unknown".
3. Trim whitespace.
4. Remove leading * from club names.
5. Keep non-ASCII characters because this is international data.

In [0]:
# Count missing athlete_club values
athlete_club_null_count = (
    marathon_silver_df
    .filter(col("athlete_club").isNull())
    .count()
)

print(f"Null athlete_club rows: {athlete_club_null_count}")

In [0]:
# Clean athlete_club more robustly:
# - replace null with "unknown"
# - trim whitespace first
# - remove one or more leading "*" characters
# - trim again after removing "*"
marathon_silver_df = marathon_silver_df.withColumn(
    "athlete_club",
    trim(
        regexp_replace(
            trim(coalesce(col("athlete_club"), lit("unknown"))),
            r"^\*+\s*",
            ""
        )
    )
)

# Replace empty values with "unknown"
marathon_silver_df = marathon_silver_df.withColumn(
    "athlete_club",
    when(length(trim(col("athlete_club"))) == 0, lit("unknown"))
    .otherwise(col("athlete_club"))
)

#### verify the cleaning

In [0]:
# Check null athlete_club values after cleaning
athlete_club_null_count_after = (
    marathon_silver_df
    .filter(col("athlete_club").isNull())
    .count()
)

print(f"Null athlete_club rows after cleaning: {athlete_club_null_count_after}")

In [0]:
# Check if any athlete_club values still start with "*"
athlete_club_star_count_after = (
    marathon_silver_df
    .filter(col("athlete_club").rlike(r"^\*"))
    .count()
)

print(f"Athlete club rows still starting with *: {athlete_club_star_count_after}")

#### Inspect those 4 remaing with *

In [0]:
# Inspect the remaining athlete_club values that still start with "*"
display(
    marathon_silver_df
    .filter(col("athlete_club").rlike(r"^\*"))
    .select(
        "athlete_club",
        "event_name",
        "year_of_event",
        "athlete_country",
        "athlete_id"
    )
)

- Convert "***" club values to "unknown".

In [0]:
# Remove leading stars and convert empty club values to "unknown"
marathon_silver_df = marathon_silver_df.withColumn(
    "athlete_club",
    trim(regexp_replace(col("athlete_club"), r"^\*+", ""))
)

marathon_silver_df = marathon_silver_df.withColumn(
    "athlete_club",
    when(length(trim(col("athlete_club"))) == 0, lit("unknown"))
    .otherwise(col("athlete_club"))
)

In [0]:
# Confirm no club values still start with "*"
athlete_club_star_count_after = (
    marathon_silver_df
    .filter(col("athlete_club").rlike(r"^\*"))
    .count()
)

# Confirm no null club values remain
athlete_club_null_count_after = (
    marathon_silver_df
    .filter(col("athlete_club").isNull())
    .count()
)

print(f"Athlete club rows still starting with *: {athlete_club_star_count_after}")
print(f"Null athlete_club rows after cleaning: {athlete_club_null_count_after}")

#### display result

In [0]:
# Display most common cleaned athlete_club values
display(
    marathon_silver_df
    .groupBy("athlete_club")
    .count()
    .orderBy(col("count").desc())
)

#### Results:
- The athlete_club column was cleaned by replacing missing values with "unknown", trimming whitespace, and removing leading * characters.
- Rows where the club value only contained stars, such as ***, were converted to "unknown".
- After cleaning, there are no null athlete_club values and no club values starting with *.

## 3. `athlete_country`
**Raw EDA finding:**
- 3 null values
- 209 distinct values
- country codes are mostly 3-letter codes
- null rows can be removed in Silver

**Silver rule for athlete_country**
1. Trim whitespace
2. Convert to uppercase
3. Remove rows where athlete_country is null
4. Later join with country mapping table to add country_name


#### check for Nulls if they still exist

In [0]:
# Check missing athlete_country values after previous Silver cleaning steps
athlete_country_null_count = (
    marathon_silver_df
    .filter(col("athlete_country").isNull())
    .count()
)

print(f"Null athlete_country rows: {athlete_country_null_count}")

In [0]:
# Inspect rows where athlete_country is missing
display(
    marathon_silver_df
    .filter(col("athlete_country").isNull())
    .select(
        "year_of_event",
        "event_dates",
        "event_name",
        "event_distance_length",
        "athlete_performance",
        "athlete_club",
        "athlete_country",
        "athlete_id"
    )
)

#### remove that NULL value

In [0]:
# Standardize athlete_country and remove missing values
rows_before_country_cleaning = marathon_silver_df.count()

marathon_silver_df = (
    marathon_silver_df
    .withColumn("athlete_country", upper(trim(col("athlete_country"))))
    .filter(col("athlete_country").isNotNull())
)

rows_after_country_cleaning = marathon_silver_df.count()

print(f"Rows before athlete_country cleaning: {rows_before_country_cleaning}")
print(f"Rows after athlete_country cleaning: {rows_after_country_cleaning}")
print(f"Rows removed: {rows_before_country_cleaning - rows_after_country_cleaning}")

#### verify

In [0]:
# Confirm no missing athlete_country values remain
athlete_country_null_count_after = (
    marathon_silver_df
    .filter(col("athlete_country").isNull())
    .count()
)

print(f"Null athlete_country rows after cleaning: {athlete_country_null_count_after}")

In [0]:
# Display cleaned athlete_country values
display(
    marathon_silver_df
    .select(
        "athlete_country",
        "event_name",
        "year_of_event",
        "athlete_performance",
        "athlete_id"
    )
    .limit(100)
)

#### Show distinct countries

In [0]:
# Count distinct athlete country codes after cleaning
distinct_athlete_country_count = (
    marathon_silver_df
    .select("athlete_country")
    .distinct()
    .count()
)

print(f"Distinct athlete_country values after cleaning: {distinct_athlete_country_count}")

#### Most represented countries

In [0]:
# Show most represented athlete countries after cleaning
display(
    marathon_silver_df
    .groupBy("athlete_country")
    .agg(count("*").alias("row_count"))
    .orderBy(col("row_count").desc())
)

#### Results:
- The raw EDA found only a few missing values. Rows with missing athlete_country are removed in the Silver layer.
- I also standardized athlete_country by trimming whitespace and converting the values to uppercase.

## 3. `athlete_year_of_birth`

**Silver layer Rules:**
1. Cast athlete_year_of_birth to integer.
2. Keep NULL birth years as NULL.
3. Set clearly invalid birth years to NULL.
4. Calculate athlete_age_at_event.
5. Set unrealistic ages to NULL.

- Findings from raw layer eda, dataset starts from 1798, values like 1786 and 1791 are not automatically invalid. But 1193 is clearly wrong.

#### Why I keep the age with NULL Values
- Because the race result itself is still valid if I have: event, athlete_country, performance, gender, distance and speed.
- If I remove all rows where age is null, I would delete around 579,518 rows.
- That is a large amount of useful race-result data and the dashboard numbers will become smaller and slightly biased.


- Gold/dashboard layer can apply metric-specific filters like:
```
WHERE athlete_age_at_event IS NOT NULL
```

#### check current nulls and invalid values

In [0]:
# Check current birth year issues after previous Silver cleaning
birth_year_issue_df = (
    marathon_silver_df
    .filter(
        col("athlete_year_of_birth").isNull() |
        (col("athlete_year_of_birth") == 1193)
    )
    .groupBy("athlete_year_of_birth")
    .agg(count("*").alias("row_count"))
    .orderBy("athlete_year_of_birth")
)

display(birth_year_issue_df)

#### cleaning
1193 → NULL
NULL = NULL

In [0]:
# Cast birth year to integer and set clearly invalid value to NULL
marathon_silver_df = marathon_silver_df.withColumn(
    "athlete_year_of_birth",
    when(col("athlete_year_of_birth") == 1193, None)
    .otherwise(col("athlete_year_of_birth").cast("int"))
)

# Calculate athlete age at event
marathon_silver_df = marathon_silver_df.withColumn(
    "athlete_age_at_event",
    col("year_of_event") - col("athlete_year_of_birth")
)

# Set unrealistic ages to NULL, but keep the row
marathon_silver_df = marathon_silver_df.withColumn(
    "athlete_age_at_event",
    when(
        (col("athlete_age_at_event") < 5) |
        (col("athlete_age_at_event") > 100),
        None
    ).otherwise(col("athlete_age_at_event"))
)

#### verify

In [0]:
# Check birth year and calculated age after cleaning
display(
    marathon_silver_df
    .select(
        "year_of_event",
        "athlete_year_of_birth",
        "athlete_age_at_event"
    )
    .summary()
)

In [0]:
from pyspark.sql.functions import count

# Count missing birth year and missing calculated age
birth_year_age_quality_df = marathon_silver_df.select(
    count(when(col("athlete_year_of_birth").isNull(), True)).alias("null_birth_year_count"),
    count(when(col("athlete_age_at_event").isNull(), True)).alias("null_age_at_event_count")
)

display(birth_year_age_quality_df)

## 4. `athlete_age_category`

**Silver rule for athlete_age_category**
1. Keep the row.
2. Trim whitespace.
3. Convert to uppercase.
4. Replace NULL or empty values with "unknown".

- Reason is this is a categorical column, not a numeric column. For dashboards, unknown is easier to group, count, and explain. It also avoids hiding many rows when I make charts.


In [0]:
# Clean athlete_age_category:
# - trim spaces
# - standardize to uppercase
# - replace null/empty values with "unknown"
marathon_silver_df = marathon_silver_df.withColumn(
    "athlete_age_category",
    when(
        col("athlete_age_category").isNull() |
        (length(trim(col("athlete_age_category"))) == 0),
        lit("unknown")
    ).otherwise(
        upper(trim(col("athlete_age_category")))
    )
)

#### verify cleaning results

In [0]:
# Check missing athlete_age_category after cleaning
athlete_age_category_null_count = (
    marathon_silver_df
    .filter(col("athlete_age_category").isNull())
    .count()
)

print(f"Null athlete_age_category rows after cleaning: {athlete_age_category_null_count}")

In [0]:
# Display athlete_age_category distribution after cleaning
display(
    marathon_silver_df
    .groupBy("athlete_age_category")
    .agg(count("*").alias("row_count"))
    .orderBy(col("row_count").desc())
)

#### Results:
- The athlete_age_category column contains missing values. Since age category is useful for grouping but is not required to identify a valid race result, I do not remove rows where this value is missing.
- In the Silver layer, I standardize athlete_age_category by trimming whitespace, converting values to uppercase, and replacing missing values with "unknown".
- This makes the column easier to use in dashboards because unknown age categories can be grouped and counted explicitly.

## 5. `athlete_gender`

Findings from the raw eda layer:
- Null gender rows: 7
- Gender values: M, F, X, null
- Decision: remove null gender rows

#### check remaining gender column values

In [0]:
# Check current athlete_gender distribution after previous Silver cleaning
display(
    marathon_silver_df
    .groupBy("athlete_gender")
    .agg(count("*").alias("row_count"))
    .orderBy(col("row_count").desc())
)

#### clean gender text and remove nulls

In [0]:
# Standardize athlete_gender and remove missing values
rows_before_gender_cleaning = marathon_silver_df.count()

marathon_silver_df = (
    marathon_silver_df
    .withColumn("athlete_gender", upper(trim(col("athlete_gender"))))
    .filter(col("athlete_gender").isNotNull())
)

rows_after_gender_cleaning = marathon_silver_df.count()

print(f"Rows before athlete_gender cleaning: {rows_before_gender_cleaning}")
print(f"Rows after athlete_gender cleaning: {rows_after_gender_cleaning}")
print(f"Rows removed: {rows_before_gender_cleaning - rows_after_gender_cleaning}")

#### verify cleaned results

In [0]:
# Confirm no null athlete_gender values remain
athlete_gender_null_count_after = (
    marathon_silver_df
    .filter(col("athlete_gender").isNull())
    .count()
)

print(f"Null athlete_gender rows after cleaning: {athlete_gender_null_count_after}")

In [0]:
# Display cleaned athlete_gender distribution
display(
    marathon_silver_df
    .groupBy("athlete_gender")
    .agg(count("*").alias("row_count"))
    .orderBy(col("row_count").desc())
)

#### Results:
The athlete_gender column contains mostly M and F values, with a small number of X values and a few nulls.

In the Silver layer, I standardized athlete_gender by trimming whitespace and converting values to uppercase. Rows with missing athlete_gender were removed because they were very few and problematic in other columns.

The value X was kept as a valid gender category.

## 6. `athlete_average_speed`
**Based on the findings I had from the eda raw layer, the rules for cleaning in the silver layer are:**
- Do not trust the raw athlete_average_speed blindly. There's a lot to clean still. Like:
- Convert it to numeric type.
- Fix values that are clearly scaled wrong.
- Set unreliable zero speeds to NULL.
- Keep the row unless the row is already invalid for other reasons.

---

**More Findings:**
- athlete_average_speed is stored as string
- some values are time-like strings, such as 09:29:17
- median is around 7.354
- most normal values are around 5.783 to 9.1
- max is 29644, which is not a real speed
- values above 1000 are probably decimal separator issues
    - example: 28302 should be 28.302
- some 0.0 values are unreliable
- k/K event distance rows had athlete_average_speed = 0
- day-based and Etappen events were removed 

----

**Cleaning plan:**
1. **Convert to numeric**
- Create `athlete_average_speed_numeric` and use `try_cast` so malformed values become NULL instead of crashing. 

2. **Fix scaled values above 1000**
- If the value is greater than 1000, divide by 1000.  `28302  → 28.302`
- Create the final clean column: `athlete_average_speed_kmh`

3. **Remove zero or negative speeds**

4. **Remove NULL speed values**

-----

```
raw_null              = original value is missing  = remove
malformed_not_numeric = value cannot be converted to number   = ??? Check first
zero_or_negative      = numeric but not useful as speed   = remove

scaled_above_1000     = probably decimal issue, fixable by /1000  = divide by 1000 and keep
valid_numeric         = already usable  = keep
```

In [0]:
# Convert athlete_average_speed safely to numeric for checking only
speed_check_df = marathon_silver_df.withColumn(
    "athlete_average_speed_numeric",
    expr("try_cast(athlete_average_speed as double)")
)

# Classify raw speed values before cleaning
speed_quality_df = (
    speed_check_df
    .withColumn(
        "speed_quality_status",
        when(col("athlete_average_speed").isNull(), "raw_null")
        .when(col("athlete_average_speed_numeric").isNull(), "malformed_not_numeric")
        .when(col("athlete_average_speed_numeric") <= 0, "zero_or_negative")
        .when(col("athlete_average_speed_numeric") > 1000, "scaled_above_1000")
        .otherwise("valid_numeric")
    )
    .groupBy("speed_quality_status")
    .agg(count("*").alias("row_count"))
    .orderBy(col("row_count").desc())
)

display(speed_quality_df)

In [0]:
# Inspect non-valid speed values before cleaning
display(
    speed_check_df
    .withColumn(
        "speed_quality_status",
        when(col("athlete_average_speed").isNull(), "raw_null")
        .when(col("athlete_average_speed_numeric").isNull(), "malformed_not_numeric")
        .when(col("athlete_average_speed_numeric") <= 0, "zero_or_negative")
        .when(col("athlete_average_speed_numeric") > 1000, "scaled_above_1000")
        .otherwise("valid_numeric")
    )
    .filter(col("speed_quality_status") != "valid_numeric")
    .select(
        "event_distance_length",
        "event_distance_unit",
        "athlete_performance",
        "athlete_average_speed",
        "athlete_average_speed_numeric",
        "speed_quality_status",
        "athlete_country",
        "athlete_id"
    )
    .limit(100)
)

#### Clean and throw away rows i dont want
```
raw_null              = original value is missing  = remove
malformed_not_numeric = value cannot be converted to number   = check first??
zero_or_negative      = numeric but not useful as speed   = remove
scaled_above_1000     = probably decimal issue, fixable by /1000  = dive by 1000 and keep
valid_numeric         = already usable  = keep
```

In [0]:
# Convert athlete_average_speed safely to numeric
marathon_silver_df = marathon_silver_df.withColumn(
    "athlete_average_speed_numeric",
    expr("try_cast(athlete_average_speed as double)")
)

# Fix scaled speed values and create final speed column
marathon_silver_df = marathon_silver_df.withColumn(
    "athlete_average_speed_kmh",
    when(
        col("athlete_average_speed_numeric") > 1000,
        col("athlete_average_speed_numeric") / 1000
    ).otherwise(col("athlete_average_speed_numeric"))
)

# Remove invalid speed rows:
# - raw null
# - malformed strings
# - zero or negative values
rows_before_speed_cleaning = marathon_silver_df.count()

marathon_silver_df = marathon_silver_df.filter(
    col("athlete_average_speed_kmh").isNotNull() &
    (col("athlete_average_speed_kmh") > 0)
)

rows_after_speed_cleaning = marathon_silver_df.count()

print(f"Rows before athlete_average_speed cleaning: {rows_before_speed_cleaning}")
print(f"Rows after athlete_average_speed cleaning: {rows_after_speed_cleaning}")
print(f"Rows removed: {rows_before_speed_cleaning - rows_after_speed_cleaning}")

#### vERIFY

In [0]:
# Check cleaned speed summary
display(
    marathon_silver_df
    .select("athlete_average_speed_kmh")
    .summary()
)

In [0]:
# Confirm no invalid cleaned speeds remain
invalid_speed_remaining_count = (
    marathon_silver_df
    .filter(
        col("athlete_average_speed_kmh").isNull() |
        (col("athlete_average_speed_kmh") <= 0)
    )
    .count()
)

print(f"Invalid athlete_average_speed_kmh rows remaining: {invalid_speed_remaining_count}")

In [0]:
from pyspark.sql.functions import col, when, count

# Check speed quality after cleaning
speed_quality_after_df = (
    marathon_silver_df
    .withColumn(
        "speed_quality_status",
        when(col("athlete_average_speed").isNull(), "raw_null")
        .when(col("athlete_average_speed_numeric").isNull(), "malformed_not_numeric")
        .when(col("athlete_average_speed_numeric") <= 0, "zero_or_negative")
        .when(col("athlete_average_speed_numeric") > 1000, "scaled_above_1000")
        .otherwise("valid_numeric")
    )
    .groupBy("speed_quality_status")
    .agg(count("*").alias("row_count"))
    .orderBy(col("row_count").desc())
)

display(speed_quality_after_df)

In [0]:
# Confirm removed invalid speed categories are gone
removed_speed_categories_count = (
    marathon_silver_df
    .filter(
        col("athlete_average_speed").isNull() |
        col("athlete_average_speed_numeric").isNull() |
        (col("athlete_average_speed_numeric") <= 0)
    )
    .count()
)

print(f"Rows remaining in removed speed categories: {removed_speed_categories_count}")

In [0]:
# Display remaining invalid speed rows, if any
display(
    marathon_silver_df
    .filter(
        col("athlete_average_speed").isNull() |
        col("athlete_average_speed_numeric").isNull() |
        (col("athlete_average_speed_numeric") <= 0)
    )
    .select(
        "event_distance_length",
        "athlete_performance",
        "athlete_average_speed",
        "athlete_average_speed_numeric",
        "athlete_average_speed_kmh",
        "athlete_id"
    )
)

- The `athlete_average_speed` column was stored as a **string** and contained **malformed, zero, null, and incorrectly scaled values.**

- For simplicity in the Silver layer, I converted the column to a numeric value. Values above 1000 appeared to have a missing decimal separator, so I divided those values by 1000.

- Rows with null, malformed, zero, or negative speed values were removed because they are invalid for speed analysis.

- The final cleaned speed column is `athlete_average_speed_kmh`.

#### show what is left after athlete_average_speed cleaning

In [0]:
# Show remaining speed quality categories after cleaning
display(
    marathon_silver_df
    .withColumn(
        "speed_quality_status",
        when(col("athlete_average_speed_numeric") > 1000, "scaled_above_1000_fixed")
        .otherwise("valid_numeric")
    )
    .groupBy("speed_quality_status")
    .agg(count("*").alias("row_count"))
    .orderBy(col("row_count").desc())
)

In [0]:
# Display cleaned athlete_average_speed result
display(
    marathon_silver_df
    .select(
        "event_distance_length",
        "athlete_performance",
        "athlete_average_speed",
        "athlete_average_speed_numeric",
        "athlete_average_speed_kmh",
        "athlete_id"
    )
    .limit(100)
)

#### Results:
- The original average speed column was stored as text and had formatting issues. I converted it to a numeric helper column, fixed scaled values by dividing values above 1000 by 1000, removed invalid speeds, and created a final cleaned column called athlete_average_speed_kmh. 
- This final column is what I will use in the Gold layer and dashboard.


```
athlete_average_speed              = original raw value
athlete_average_speed_numeric      = raw value converted to number
athlete_average_speed_kmh          = final cleaned value to use later in other layers
```

##### 1. `athlete_average_speed`
- `7666.0` This is the original column from the dataset but the problem is that Spark read it as a **string not numeric.** 
- I need this column to have the tracability of the first conversion from ###h to numeric values.

##### 2. `athlete_average_speed_numeric`
- This is a temporary helper column. **It converts the original string into a number.**
- **Not the final usable column for the gold layer.**
```
athlete_average_speed         = "7666.0"
athlete_average_speed_numeric = 7666
```
- I used it to help detect these values when cleaning:
```
values > 1000 (divide by 100)
values <= 0 (removed)
malformed values (removed)
null values (removed)
```

##### 3. `athlete_average_speed_kmh`
- **Final cleaned up column from the raw dataset. Will be used later in gold layer. **
```
athlete_average_speed_numeric = 7666
athlete_average_speed_kmh     = 7.666
```
- EDA showed that values like 7666 are probably missing a decimal separator and should be 7.666.

-----

##### 4. In the Silver OBT Layer
- Keep: `athlete_average_speed_kmh` as final cleaned value
- Remove: `athlete_average_speed` and `athlete_average_speed_numeric`



## 7. `athlete_id`

#### Findings from raw eda
%md
- Null athlete_id values: 0
- Total rows: 7,461,195
- Distinct athlete IDs: 1,641,168
- Repeated athlete ID rows: 5,820,027

- **Athlete ID identifies an athlete, not a single race result.**
- **So one athlete can appear in multiple rows because they may have participated in multiple races/events over time.**

#### Dimensional Modelling for athlete

```
dim_athlete uses athlete_id
fct_results references athlete_id

dim_athlete
├── athlete_id
├── athlete_gender
├── athlete_year_of_birth
├── athlete_age_category
├── athlete_country
```
race results table can reference it:
```
fct_results
├── result_id
├── athlete_id
├── event_id
├── performance
```

#### Further checks:
1. Check nulls after previous cleaning
2. Cast athlete_id to integer
3. Keep repeated athlete_id values
4. Do not create a new athlete_id

#### check nulls again

In [0]:
# Check missing athlete_id values after Silver cleaning steps
athlete_id_null_count = (
    marathon_silver_df
    .filter(col("athlete_id").isNull())
    .count()
)

print(f"Null athlete_id rows: {athlete_id_null_count}")

In [0]:
# Check distinct athlete_id count after Silver cleaning
total_rows = marathon_silver_df.count()

distinct_athlete_id_count = (
    marathon_silver_df
    .select(countDistinct("athlete_id").alias("distinct_count"))
    .collect()[0]["distinct_count"]
)

print(f"Total rows: {total_rows}")
print(f"Distinct athlete IDs: {distinct_athlete_id_count}")
print(f"Repeated athlete ID rows: {total_rows - distinct_athlete_id_count}")

#### cast athlete_id to integer

In [0]:
# Cast athlete_id to integer for consistent joins and dimensional modelling
marathon_silver_df = marathon_silver_df.withColumn(
    "athlete_id",
    col("athlete_id").cast("int")
)

In [0]:
# Show athletes that appear in multiple rows
display(
    marathon_silver_df
    .groupBy("athlete_id")
    .agg(count("*").alias("result_count"))
    .filter(col("result_count") > 1)
    .orderBy(col("result_count").desc())
)

# Display the full athlete dataframe

In [0]:
# Display current cleaned Silver dataframe
display(marathon_silver_df.limit(100))

In [0]:
# Check final schema after event and athlete cleaning
marathon_silver_df.printSchema()

In [0]:
# Check final row count after all current cleaning
print(f"Rows after event and athlete cleaning: {marathon_silver_df.count()}")